# Hotel Booking Cancellation Prediction System
This notebook follows Steps 1–8 from the project workflow and adds a Logistic Regression baseline.
Upload `hotel_bookings.csv` to the notebook environment or update `DATA_PATH` below.


In [ ]:
!pip -q install pandas numpy matplotlib seaborn scikit-learn joblib


In [ ]:
DATA_PATH = 'hotel_bookings.csv'  # Change this path if needed
OUTPUT_DIR = 'hotel_project_output'


## Run the complete workflow
The next cell loads the CSV, performs inspection and cleaning, creates EDA plots, detects outliers, builds a leakage-aware preprocessing pipeline, performs an 80:20 stratified split, and evaluates a baseline model.


In [ ]:
"""
Hotel Booking Cancellation Prediction
Run:
    pip install pandas numpy matplotlib seaborn scikit-learn joblib
    python hotel_cancellation_project.py --data hotel_bookings.csv

Outputs are written to ./hotel_project_output/
"""
import argparse
from pathlib import Path
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

warnings.filterwarnings("ignore", category=FutureWarning)

def main(data_path):
    output = Path(OUTPUT_DIR)
    output.mkdir(exist_ok=True)

    # STEP 1: DATA COLLECTION & LOADING
    df = pd.read_csv(data_path)
    print("\\nSTEP 1 - DATA LOADING")
    print("Dataset shape:", df.shape)
    print("Columns:", df.columns.tolist())
    assert "is_canceled" in df.columns, "Target column 'is_canceled' is missing."
    assert set(df["is_canceled"].dropna().unique()).issubset({0, 1}), "Target must be binary."

    # STEP 2: INITIAL DATA INSPECTION
    print("\\nSTEP 2 - INITIAL INSPECTION")
    print("\\nFirst five rows:\\n", df.head())
    print("\\nLast five rows:\\n", df.tail())
    print("\\nDataset information:")
    df.info()
    print("\\nNumerical summary:\\n", df.describe(include="all").T)
    print("\\nMissing values:\\n", df.isna().sum().sort_values(ascending=False))
    print("\\nUnique values per column:\\n", df.nunique().sort_values())

    # STEP 3: DATA CLEANING
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(r"[^a-z0-9]+", "_", regex=True)
                  .str.strip("_"))
    duplicate_count = int(df.duplicated().sum())
    df = df.drop_duplicates().copy()
    print(f"Exact duplicate rows removed: {duplicate_count}")
    print("Shape after duplicate removal:", df.shape)

    # Missing values: children -> median; country -> Unknown.
    df["children"] = df["children"].fillna(df["children"].median())
    df["country"] = df["country"].fillna("Unknown")

    # Agent/company are identifiers, not continuous measurements.
    # Keep missing values as an explicit category.
    for col in ["agent", "company"]:
        df[col] = df[col].fillna("Unknown").astype(str)

    # Correct the single negative ADR using the median of valid non-negative ADR.
    valid_adr_median = df.loc[df["adr"] >= 0, "adr"].median()
    df.loc[df["adr"] < 0, "adr"] = valid_adr_median

    # Add interpretable booking features.
    df["total_guests"] = df[["adults", "children", "babies"]].sum(axis=1)
    df["total_stay_nights"] = (
        df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
    )

    # STEP 4: EXPLORATORY DATA ANALYSIS
    print("\\nSTEP 4 - EDA")
    print("Target counts:\\n", df["is_canceled"].value_counts())
    print("Target percentages:\\n", (df["is_canceled"].value_counts(normalize=True) * 100).round(2))
    print("\\nCancellation rate by hotel (%):\\n",
          (df.groupby("hotel")["is_canceled"].mean() * 100).round(2))
    print("\\nCancellation rate by deposit type (%):\\n",
          (df.groupby("deposit_type")["is_canceled"].mean() * 100).round(2))
    print("\\nCancellation rate by market segment (%):\\n",
          (df.groupby("market_segment")["is_canceled"].mean() * 100).round(2))

    sns.set_theme()
    plt.figure(figsize=(7, 4))
    sns.countplot(data=df, x="is_canceled")
    plt.title("Booking Cancellation Distribution")
    plt.xlabel("is_canceled (0 = Not canceled, 1 = Canceled)")
    plt.tight_layout()
    plt.savefig(output / "target_distribution.png", dpi=150)
    plt.close()

    plt.figure(figsize=(8, 4))
    sns.histplot(data=df, x="lead_time", hue="is_canceled",
                 bins=40, element="step", stat="density", common_norm=False)
    plt.title("Lead Time Distribution by Cancellation")
    plt.tight_layout()
    plt.savefig(output / "lead_time_distribution.png", dpi=150)
    plt.close()

    hotel_rate = (df.groupby("hotel")["is_canceled"].mean() * 100).sort_values()
    plt.figure(figsize=(7, 4))
    hotel_rate.plot(kind="bar")
    plt.ylabel("Cancellation rate (%)")
    plt.title("Cancellation Rate by Hotel Type")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(output / "cancellation_by_hotel.png", dpi=150)
    plt.close()

    # STEP 5: OUTLIER DETECTION & TREATMENT
    print("\\nSTEP 5 - OUTLIER CHECK (IQR)")
    outlier_rows = []
    for col in ["lead_time", "adr", "days_in_waiting_list", "stays_in_week_nights"]:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        count = int(((df[col] < lower) | (df[col] > upper)).sum())
        outlier_rows.append((col, round(lower, 2), round(upper, 2), count))
    outlier_table = pd.DataFrame(
        outlier_rows, columns=["feature", "IQR_lower", "IQR_upper", "outlier_count"]
    )
    print(outlier_table.to_string(index=False))
    outlier_table.to_csv(output / "outlier_summary.csv", index=False)

    # These high values may be genuine booking patterns, so rows are not deleted.
    # Apply log1p to right-skewed, non-negative numeric variables.
    for col in ["lead_time", "adr", "days_in_waiting_list"]:
        df[col] = np.log1p(df[col].clip(lower=0))

    # STEP 6: FEATURE ENCODING
    # OneHotEncoder is used for nominal categories; no arbitrary label order is imposed.
    # The encoder is fitted inside the pipeline using training data only.

    # STEP 7: FEATURE SCALING
    # StandardScaler is applied to numeric columns inside the pipeline.
    # It is fitted only on the training partition to prevent data leakage.

    # Remove post-outcome / leakage columns:
    # reservation_status directly reveals the outcome; reservation_status_date
    # is recorded after the booking outcome; assigned_room_type may be assigned later.
    leakage_cols = ["reservation_status", "reservation_status_date", "assigned_room_type"]
    X = df.drop(columns=["is_canceled"] + leakage_cols)
    y = df["is_canceled"]

    categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
    numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), numerical_cols),
            ("categorical", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10))
            ]), categorical_cols)
        ],
        remainder="drop"
    )

    # STEP 8: TRAIN-TEST SPLIT (80:20, stratified)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )
    print("\\nSTEP 8 - TRAIN/TEST SPLIT")
    print("Training rows:", len(X_train), "Testing rows:", len(X_test))
    print("Training target ratio:\\n", y_train.value_counts(normalize=True).round(3))
    print("Testing target ratio:\\n", y_test.value_counts(normalize=True).round(3))

    # STEP 9: BASELINE CLASSIFICATION MODEL (project extension)
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000, class_weight="balanced", solver="liblinear"
        ))
    ])
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    print("\\nSTEP 9 - BASELINE MODEL EVALUATION")
    metrics = {
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1_score": f1_score(y_test, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probabilities)
    }
    for name, value in metrics.items():
        print(f"{name}: {value:.4f}")
    print("\\nConfusion matrix:\\n", confusion_matrix(y_test, predictions))
    print("\\nClassification report:\\n",
          classification_report(y_test, predictions, zero_division=0))

    pd.DataFrame([metrics]).to_csv(output / "model_metrics.csv", index=False)
    joblib.dump(model, output / "hotel_cancellation_model.joblib")
    print(f"\\nCompleted. Files saved in: {output.resolve()}")

main(DATA_PATH)
